In [ ]:
!nvidia-smi

In [ ]:

!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

In [ ]:

import torch
import transformers
import datasets
import peft #Parameter-Efficient Fine-Tuning
import trl  #Transformer Reinforcement Learning, It provides high-level tools like
            #the SFTTrainer (Supervised Fine-Tuning Trainer) to easily combine the dataset,
            #  the compressed model, and the PEFT adapters into a streamlined training loop
import bitsandbytes #Quantization, It compresses large 16-bit or 32-bit models down into 8-bit or 4-bit sizes
import accelerate

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Accelerate:", accelerate.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Colab, select a GPU runtime."
    )

print("GPU:", torch.cuda.get_device_name(0))

total_vram = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"Total VRAM: {total_vram:.2f} GB")

free_vram, total_vram = torch.cuda.mem_get_info()

print(f"Free VRAM: {free_vram / (1024 ** 3):.2f} GB")
print(f"Total VRAM: {total_vram / (1024 ** 3):.2f} GB")

!nvidia-smi

In [ ]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Load the tokenizer for the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Setting up the model with 4-bit quantization using BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("4-bit compute dtype:", bnb_config.bnb_4bit_compute_dtype)

print(f"model name: {model_name}")


In [ ]:
from transformers import AutoModelForCausalLM

qlora_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
print(f"qlora model: {qlora_model}")
print(f"qlora model device: {qlora_model.device}")

free_vram, total_vram  = torch.cuda.mem_get_info()
print(f"Free VRAM: {free_vram / (1024 ** 3):.2f} GB")
print(f"Total VRAM: {total_vram / (1024 ** 3):.2f} GB")

In [ ]:
total_par = sum(
    p.numel()
    for p in qlora_model.parameters()
)

print("\nModel parameters:")
print(f"Total parameters: {total_par:,}")
print(f"Total parameters: {total_par / 1e9:.2f}B")

# Check how much GPU memory the 4-bit model uses
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)

    print("\nQLoRA 4-bit GPU memory:")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved:  {reserved:.2f} GB")

In [ ]:
# Check the actual model configuration

print("Model name:", model_name)
print("Architecture:", qlora_model.config.architectures)
print("Hidden size:", qlora_model.config.hidden_size)
print("Number of layers:", qlora_model.config.num_hidden_layers)
print("Vocab size:", qlora_model.config.vocab_size)

print("\nParameter count:")
print(sum(p.numel() for p in qlora_model.parameters()))
print(qlora_model)

In [ ]:
from peft import LoraConfig, get_peft_model
from peft import prepare_model_for_kbit_training

#prepare the model for k-bit trainning

qLora_base_model = prepare_model_for_kbit_training(qlora_model)

#lora adopter config

lora_config = LoraConfig(
    r=16, # rank is the number of parameters in the low-rank matrices used to approximate the original weight matrices. A higher rank allows for a more paramter trainable (expressive) model, but also increases the number of parameters and computational cost.
    lora_alpha = 32, # Lora_alpha is a hyperparameter that controls the scalibility of the low-rank matrices
    lora_dropout = 0.05, # Lora_dropout is a regularization technique that randomly sets a fraction of the low-rank matrix elements to zero during training, which helps prevent overfitting

    target_modules =[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias = "none",
    task_type = "CAUSAL_LM"
)



In [ ]:
#adding lora adopter to the 4-bit model

qlora_model = get_peft_model(
    qLora_base_model,
    lora_config
)

In [ ]:
# baba blackship nah just kidding let's see the trainable parameters of the model
qlora_model.print_trainable_parameters()

In [ ]:

from datasets import load_dataset, Dataset

dataset = load_dataset("raghu298/ml-interview-sft-dataset")

print("Original dataset:")
print(dataset)

# checking for duplicate questions
questions = [
    example["question"].strip().lower()
    for example in dataset["train"]
]

print("\nOriginal examples:", len(questions))
print("Unique questions:", len(set(questions)))
print("Duplicate questions:", len(questions) - len(set(questions)))


#

In [ ]:

unique_data = []
seen_questions = set()

for example in dataset["train"]:
    question = example["question"].strip().lower()

    if question not in seen_questions:
        seen_questions.add(question)
        unique_data.append(example)

clean_dataset = Dataset.from_list(unique_data)

print("\nAfter deduplication:", len(clean_dataset))
print("Removed:", len(dataset["train"]) - len(clean_dataset))

In [ ]:

split_1 = clean_dataset.train_test_split(
    test_size=0.10,
    seed=42
)

train_val = split_1["train"]
test_dataset = split_1["test"]

#    10% of the original dataset ≈ validation
split_2 = train_val.train_test_split(
    test_size=0.1111,
    seed=42
)

train_dataset = split_2["train"]
val_dataset = split_2["test"]

# just verifying
print("\nFinal dataset:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nExample:")
print(train_dataset[0])

In [ ]:
#setting up the data for sft 🫂

def format_example(example):
    return{
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize = False
        )
    }

train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)
test_dataset = test_dataset.map(format_example)

print("Formatted dataset:")
print(train_dataset[0]["text"])

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/content/qlora-interview-coach",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    gradient_checkpointing=False,

    report_to="none",

    max_length=512,
    dataset_text_field="text"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=qlora_model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    processing_class=tokenizer
)

print("QLoRA trainer recreated successfully.")

In [ ]:

print(
    "QLoRA compute dtype:",
    qlora_model.config.quantization_config.bnb_4bit_compute_dtype
)

In [ ]:
trainer.train()

In [ ]:
qlora_model.save_pretrained("/content/ai-interview-coach-qlora")
tokenizer.save_pretrained("/content/ai-interview-coach-qlora")

In [ ]:
# test_loss and perplexity just tell how well the next token is produced by comparing the test data set

import math

print("QLORA TEST EVALUATION")


eval_results = trainer.evaluate(
    eval_dataset=test_dataset
)

test_loss = eval_results["eval_loss"]
test_perplexity = math.exp(test_loss)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Perplexity: {test_perplexity:.2f}")

In [ ]:
# compring the base model and the qlora model
def generate_answer(model, tokenizer, question, max_new_tokens=150):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert AI/ML interview coach. "
                "Provide a clear, technically accurate interview answer."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


print("=" * 60)
print("BASE vs QLORA")
print("=" * 60)

for i, example in enumerate(test_dataset.select(range(3))):

    question = example["question"]

    with qlora_model.disable_adapter():
        base_answer = generate_answer(
            qlora_model,
            tokenizer,
            question
        )

    qlora_answer = generate_answer(
        qlora_model,
        tokenizer,
        question
    )

    print("\n" + "=" * 80)
    print(f"QUESTION {i+1}:")
    print(question)

    print("\nBASE:")
    print(base_answer)

    print("\nQLORA:")
    print(qlora_answer)

In [ ]:
!pip install -q evaluate rouge_score

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
predictions_base =[]
prediction_qlora =[]
references =[]

eval_samples =  test_dataset.select(
    range(min(20, len(test_dataset)))
)

for example in eval_samples:
    question = example["question"]
    reference_answer = example["answer"]

    with qlora_model.disable_adapter():
        base_answer = generate_answer(
            qlora_model,
            tokenizer,
            question
        )

    qlora_answer = generate_answer(
        qlora_model,
        tokenizer,
        question
    )

    predictions_base.append(base_answer)
    prediction_qlora.append(qlora_answer)
    references.append(reference_answer)

base_rouge = rouge.compute(
    predictions=predictions_base,
    references=references
)

qlora_rouge = rouge.compute(
    predictions=prediction_qlora,
    references=references
)
print("ROUGE-L COMPARISON")
print(f"Base ROUGE-L:  {base_rouge['rougeL']:.4f}")
print(f"QLoRA ROUGE-L: {qlora_rouge['rougeL']:.4f}")

delta = qlora_rouge["rougeL"] - base_rouge["rougeL"]

print(f"ROUGE-L change: {delta:+.4f}")

In [ ]:
import time

def benchmark_qlora(
    model,
    tokenizer,
    dataset,
    n_samples=10,
    max_new_tokens=100
):

    model.eval()

    times = []

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    samples = dataset.select(
        range(min(n_samples, len(dataset)))
    )

    for example in samples:

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert AI/ML interview coach. "
                    "Provide a clear, technically accurate interview answer."
                )
            },
            {
                "role": "user",
                "content": example["question"]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.no_grad():
            model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )

        torch.cuda.synchronize()

        end = time.perf_counter()

        times.append(end - start)

    avg_time = sum(times) / len(times)

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    return avg_time, 1 / avg_time, peak_vram


avg_time, throughput, peak_vram = benchmark_qlora(
    qlora_model,
    tokenizer,
    test_dataset
)

print("=" * 60)
print("QLORA EFFICIENCY")
print("=" * 60)

print(f"Average inference time: {avg_time:.2f} sec")
print(f"Throughput: {throughput:.4f} samples/sec")
print(f"Peak VRAM: {peak_vram:.2f} GB")

In [ ]:

from transformers import GenerationConfig, ContinuousBatchingConfig
import time
import torch


def benchmark_continuous_batching(
    model,
    tokenizer,
    dataset,
    n_requests=6,
    max_new_tokens=100
):

    model.eval()


    samples = dataset.select(
        range(min(n_requests, len(dataset)))
    )

    prompts = []

    for example in samples:

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert AI/ML interview coach. "
                    "Provide a clear, technically accurate interview answer."
                )
            },
            {
                "role": "user",
                "content": example["question"]
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Convert prompt to token IDs
        input_ids = tokenizer.encode(
            prompt,
            add_special_tokens=False
        )

        prompts.append(input_ids)

    # Generation configuration

    generation_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    # Continuous batching configuration

    continuous_batching_config = ContinuousBatchingConfig(
    max_memory_percent=0.80,
    block_size=256,
    scheduler_type="prefill_first",
    max_requests_per_batch=n_requests,
    use_cuda_graph=True,
    use_async_batching=True,
    allow_block_sharing=True
    )


    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    torch.cuda.synchronize()

    start = time.perf_counter()

#c_b
    outputs = model.generate_batch(
    prompts,
    generation_config=generation_config,
    continuous_batching_config=continuous_batching_config
    )

    torch.cuda.synchronize()

    end = time.perf_counter()
#metrices
    total_time = end - start

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    throughput = len(prompts) / total_time

    average_time = total_time / len(prompts)
    print("CONTINUOUS BATCHING")

    print(f"Requests:              {len(prompts)}")
    print(f"Total time:            {total_time:.2f} sec")
    print(f"Average time/request:  {average_time:.2f} sec")
    print(f"Throughput:            {throughput:.4f} requests/sec")
    print(f"Peak VRAM:             {peak_vram:.2f} GB")

    return outputs

In [ ]:

outputs = benchmark_continuous_batching(
    qlora_model,
    tokenizer,
    test_dataset,
    n_requests=500,
    max_new_tokens=100
)

In [ ]:
from datasets import concatenate_datasets

benchmark_dataset = concatenate_datasets(
    [test_dataset] * 19
).shuffle(seed=42).select(range(1000))

print(len(benchmark_dataset))

In [ ]:
for n in [100, 500, 750, 1000]:

    print(f"TEST: {n} REQUESTS")

    benchmark_continuous_batching(
        qlora_model,
        tokenizer,
        benchmark_dataset,
        n_requests=n,
        max_new_tokens=100
    )

In [ ]:
def generate_question(model, tokenizer, topic="RAG", difficulty="medium"):

    prompt = f"""
You are conducting a technical interview for a fresher AI Engineer.

Topic: {topic}
Difficulty: {difficulty}

Generate EXACTLY ONE interview question.

Rules:
- Ask only ONE question.
- Do not ask multiple questions.
- Do not add a second question.
- Do not provide an answer.
- Do not provide an explanation.
- Do not use headings.
- Do not use bullet points.
- The question must be clear and technically relevant.
- The question should be appropriate for a fresher AI Engineer.
- Return ONLY the question.
"""

    messages = [
        {
            "role": "system",
            "content": "You are a professional AI Engineer interviewer."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=True
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    question = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Remove markdown formatting
    question = question.replace("**", "").strip()

    return question

In [ ]:
def evaluate_answer(model, tokenizer, question, answer):

    # Handle non-answers directly
    non_answers = {
        "dk",
        "don't know",
        "dont know",
        "i don't know",
        "i dont know",
        "idk",
        "no idea",
        "not sure",
        "i'm not sure",
        "im not sure",
        "skip",
        "pass"
    }

    if answer.lower().strip() in non_answers:

        return """SCORE: 0/10

FEEDBACK:
You did not attempt the question.

WHAT WAS GOOD:
No significant technical points were provided.

WHAT WAS MISSING:
You needed to provide a technical explanation related to the question.

IDEAL ANSWER:
Try to explain the main concept first, then describe how it works and give a simple example if possible.

LEARNING POINT:
When you don't know an interview question, try to explain any related concept you know instead of immediately giving up.
"""

    prompt = f"""
You are a strict but fair AI Engineer interviewer evaluating a fresher.

INTERVIEW QUESTION:
{question}

CANDIDATE ANSWER:
{answer}

Evaluate ONLY the candidate's answer to the interview question.

Important rules:
- Do not assume the candidate said something they did not say.
- Do not give credit for information that is not present in the answer.
- Give 0/10 if the answer is completely irrelevant.
- Give a low score if the answer is incomplete.
- Give a high score only when the answer is technically correct, relevant, and sufficiently complete.
- Do not be overly generous.
- Do not invent strengths.

Return EXACTLY this format:

SCORE: X/10

FEEDBACK:
Give concise and honest feedback.

WHAT WAS GOOD:
Mention only technically correct points actually present in the candidate's answer.

WHAT WAS MISSING:
Mention important concepts that were missing.

IDEAL ANSWER:
Give a concise, technically accurate answer suitable for a fresher AI Engineer interview.

LEARNING POINT:
Teach ONE important concept from this question in simple terms.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict but helpful AI Engineer interviewer. "
                "Evaluate answers honestly and accurately."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            use_cache=True
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    evaluation = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return evaluation

In [ ]:
interview_session(
    qlora_model,
    tokenizer,
    topic="RAG",
    difficulty="medium"
)